# Stage 3B — Frozen Modeling and Internal Validation Pipeline

**Notebook version 1.2 LOCAL — Windows/Linux local execution with scikit-learn 1.8–1.9.**

This local version uses the scikit-learn 1.8+ `l1_ratio`/`C` API exclusively, contains no deprecated `penalty=` estimator argument, uses local filesystem paths, controls nested CPU parallelism, and invalidates checkpoints created by older pipeline versions.

This notebook implements the frozen Stage 3 protocol for three separate outcomes:

- `NegRFandACCP`
- `NoHandJointInvolvement`
- `AsymmetricArthritis`

## Prespecified model hierarchy

1. **Confirmatory:** common-core ridge logistic regression.
2. **Benchmark:** ordinary logistic regression with the identical core.
3. **Exploratory:** broad ridge logistic regression.
4. **Exploratory:** broad elastic-net logistic regression.

## Validation

- Outer repeated stratified five-fold cross-validation, repeated 10 times.
- Inner stratified five-fold cross-validation for tuning.
- Log loss is the tuning objective.
- All preprocessing is fitted inside the relevant training fold.
- Every development-performance prediction is out-of-fold.
- Each patient’s 10 out-of-fold probabilities are averaged.
- Confidence intervals use 2,000 stratified patient-level bootstrap resamples.

## Explainability

Linear-model SHAP contributions are calculated for outer-test patients only, centered on the corresponding outer-training transformed-feature means. Encoded columns are aggregated back to original clinical variables. Confirmatory and exploratory explanations remain separate.

> This notebook estimates internal validity for the source population. It does not provide external validation and does not establish causality.


In [ ]:
# 1. Imports and reproducible runtime settings
from __future__ import annotations

import hashlib
import json
import math
import os
import pickle
import platform
import subprocess
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from packaging.version import Version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import display, Markdown
from scipy.special import expit, logit
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss, roc_auc_score,
    roc_curve, precision_recall_curve, confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
import sklearn

warnings.filterwarnings('ignore', category=ConvergenceWarning)
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 160)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.5f}')
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')

MASTER_SEED=20260927
np.random.seed(MASTER_SEED)
print('Python:',sys.version.split()[0])
print('Platform:',platform.platform())
print('pandas:',pd.__version__)
SKLEARN_VERSION=Version(sklearn.__version__)
if SKLEARN_VERSION < Version('1.8'):
    raise RuntimeError(
        f'This local notebook requires scikit-learn >=1.8,<1.10; found {sklearn.__version__}. '
        'Install with: python -m pip install \"scikit-learn>=1.8,<1.10\"'
    )
print('scikit-learn:',sklearn.__version__)
print('LogisticRegression API mode: 1.8+ l1_ratio/C API (no deprecated penalty argument)')


In [ ]:
# 2. Local project paths and execution mode
NOTEBOOK_NAME='03B_modeling_validation_LOCAL_sklearn18.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd()
ENV_PROJECT=Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else None

# No Google Drive mount and no Colab dependency in this notebook.
# RA_PROJECT_DIR may be used if the project is not at the authorized RA_PROJECT_DIR.
candidate_roots=[p for p in [
    ENV_PROJECT,
    ENV_PROJECT,
    ENV_PROJECT/'stage2_outputs',
    Path.cwd(),
    Path.cwd()/'stage2_outputs',
    Path.cwd()/'ra_analysis'/'stage2_outputs',
    Path.cwd()/'stage2_outputs',  # portable validation fallback
] if p is not None]
PROJECT_DIR=next((p for p in candidate_roots if (p/'cleaned_data'/'cleaned_master_data.csv').exists()),None)

if PROJECT_DIR is None:
    checked='\n'.join(str(p/'cleaned_data'/'cleaned_master_data.csv') for p in candidate_roots)
    raise FileNotFoundError(
        'Could not locate Stage 2 products. Put the project at the authorized RA_PROJECT_DIR, '
        'or set the RA_PROJECT_DIR environment variable. Checked:\n'+checked
    )

protocol_candidates=[
    ENV_PROJECT/'config'/'stage3_modeling_decisions.yaml',
    ENV_PROJECT/'stage3_modeling_decisions.yaml',
    PROJECT_DIR/'config'/'stage3_modeling_decisions.yaml',
    PROJECT_DIR/'stage3_modeling_decisions.yaml',
    PROJECT_DIR.parent/'stage3_modeling_decisions.yaml',
    Path.cwd()/'stage3_modeling_decisions.yaml',
    Path.cwd()/'ra_analysis'/'stage3_modeling_decisions.yaml',
    ENV_PROJECT/'stage3_modeling_decisions.yaml',
]
PROTOCOL_PATH=next((p for p in protocol_candidates if p.exists()),None)
if PROTOCOL_PATH is None:
    raise FileNotFoundError('stage3_modeling_decisions.yaml was not found.')

DERIVED_DIR=PROJECT_DIR/'derived_data'
CLEAN_PATH=PROJECT_DIR/'cleaned_data'/'cleaned_master_data.csv'
STAGE2_AUDIT=PROJECT_DIR/'preprocessing_outputs'/'tables'/'stage2_validation_audit.csv'
STAGE3A_AUDIT=PROJECT_DIR/'modeling_feasibility'/'tables'/'stage3a_validation_audit.csv'
OUT_DIR=PROJECT_DIR/'modeling_results'
TABLE_DIR=OUT_DIR/'tables'; FIGURE_DIR=OUT_DIR/'figures'; CHECKPOINT_DIR=OUT_DIR/'checkpoints'; REPORT_DIR=OUT_DIR/'report'
for d in [OUT_DIR,TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,REPORT_DIR]: d.mkdir(parents=True,exist_ok=True)

# Full mode is the scientific analysis. Quick mode exists only for software smoke-testing.
QUICK_MODE=os.environ.get('RA_STAGE3B_QUICK_TEST','0')=='1'
FORCE_RECOMPUTE=os.environ.get('RA_STAGE3B_FORCE_RECOMPUTE','0')=='1'
if QUICK_MODE:
    OUTER_SPLITS,OUTER_REPEATS,INNER_SPLITS,BOOTSTRAPS=3,1,2,50
    RIDGE_C_GRID=np.logspace(-2,2,3)
    ENET_C_GRID=np.logspace(-2,1,3)
    ENET_L1_GRID=[0.1,0.9]
else:
    OUTER_SPLITS,OUTER_REPEATS,INNER_SPLITS,BOOTSTRAPS=5,10,5,2000
    RIDGE_C_GRID=np.logspace(-4,4,13)
    ENET_C_GRID=np.logspace(-4,2,10)
    ENET_L1_GRID=[0.1,0.5,0.9]

N_JOBS=-1
PIPELINE_MEMORY=None

OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']
CORE_PREDICTORS=['Age','Gender','RADurationDays','DASJointCountTEN28','DASJointCountSW','PatientGlobalAssessment','AbnormalCRPorESR']
MODEL_ORDER=['core_ridge','core_logistic','broad_ridge','broad_elastic_net']
print('Project:',PROJECT_DIR.resolve())
print('Protocol:',PROTOCOL_PATH.resolve())
print('Execution mode:','QUICK SOFTWARE TEST' if QUICK_MODE else 'FULL SCIENTIFIC ANALYSIS')
print('Outer:',OUTER_SPLITS,'x',OUTER_REPEATS,'| Inner:',INNER_SPLITS,'| Bootstraps:',BOOTSTRAPS)
print('Logical CPU cores:',os.cpu_count(),'| GridSearch n_jobs:',N_JOBS)
print('Pipeline caching: disabled for reliable multiprocessing')
print('Compute note: scikit-learn LogisticRegression and GridSearchCV use CPU, not CUDA.')


## Protocol and prerequisite validation

In [ ]:
# 3. Stop if frozen protocol or prior-stage audits are inconsistent
with open(PROTOCOL_PATH,'r',encoding='utf-8') as f:
    protocol=yaml.safe_load(f)

required_protocol_checks={
    'ridge primary':protocol['model_hierarchy']['confirmatory_primary']['model']=='ridge logistic regression',
    'log loss tuning':'log loss' in protocol['tuning']['primary_objective'],
    'no threshold':protocol['thresholds']['primary_threshold'] is None,
    'OOF SHAP':'out-of-fold SHAP' in protocol['explainability']['method'],
    'core predictors':protocol['common_core_predictors']==CORE_PREDICTORS,
}
if not all(required_protocol_checks.values()):
    raise AssertionError(f'Protocol mismatch: {required_protocol_checks}')

for audit_path,label in [(STAGE2_AUDIT,'Stage 2'),(STAGE3A_AUDIT,'Stage 3A')]:
    if not audit_path.exists(): raise FileNotFoundError(f'{label} audit missing: {audit_path}')
    prior=pd.read_csv(audit_path)
    if prior.empty or not prior.status.eq('PASS').all():
        raise AssertionError(f'{label} did not pass every audit check.')

clean=pd.read_csv(CLEAN_PATH)
registry=pd.read_csv(DERIVED_DIR/'dataset_registry.csv')
assert len(clean)==820 and clean.VisitID.is_unique
assert set(registry.outcome)==set(OUTCOMES)
print('Frozen protocol and prerequisite audits accepted.')


In [ ]:
# 4. Run signature protects checkpoint integrity
def sha256_file(path:Path)->str:
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

signature_payload={
    'protocol_sha256':sha256_file(PROTOCOL_PATH),
    'clean_data_sha256':sha256_file(CLEAN_PATH),
    'sklearn_version':sklearn.__version__,
    'quick_mode':QUICK_MODE,
    'outer_splits':OUTER_SPLITS,'outer_repeats':OUTER_REPEATS,'inner_splits':INNER_SPLITS,
    'ridge_grid':RIDGE_C_GRID.tolist(),'enet_c_grid':ENET_C_GRID.tolist(),'enet_l1_grid':ENET_L1_GRID,
    'master_seed':MASTER_SEED,
    'logistic_api_mode':'1.8+ l1_ratio/C',
    'pipeline_version':'3B-1.2-local-sklearn18',
}
RUN_SIGNATURE=hashlib.sha256(json.dumps(signature_payload,sort_keys=True).encode()).hexdigest()
with open(OUT_DIR/'run_signature.json','w',encoding='utf-8') as f:
    json.dump({'run_signature':RUN_SIGNATURE,**signature_payload},f,indent=2)
print('Run signature:',RUN_SIGNATURE)


## Leakage-safe preprocessing

In [ ]:
# 5. Fixed missingness transformer: creates indicators inside the pipeline
class MissingnessFlags(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        self.n_features_in_=X.shape[1]
        return self
    def transform(self,X):
        return pd.isna(X).astype(float).to_numpy() if hasattr(X,'to_numpy') else np.isnan(X).astype(float)
    def get_feature_names_out(self,input_features=None):
        if input_features is None: input_features=[f'x{i}' for i in range(self.n_features_in_)]
        return np.asarray([f'{x}__missing' for x in input_features],dtype=object)

CATEGORY_LEVELS={
    'Gender':['Female','Male'],
    'RAOnset':['Insidious','Acute','Unknown'],
    'PhysicalActivity':['Moderate','Low','High'],
}

def prepare_analysis_data(outcome:str,scope:str):
    path=DERIVED_DIR/f'{outcome}__primary_eligible.csv'
    data=pd.read_csv(path)
    manifest=pd.read_csv(DERIVED_DIR/f'{outcome}__predictor_manifest.csv')
    if scope=='core':
        predictors=CORE_PREDICTORS.copy()
    elif scope=='broad':
        predictors=manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
    else: raise ValueError(scope)
    # Fixed mathematical transformation; no parameter is learned from the data.
    data['RADurationDays_log1p']=np.log1p(data['RADurationDays'])
    if 'RADurationDays' in predictors:
        predictors=[('RADurationDays_log1p' if x=='RADurationDays' else x) for x in predictors]
    semantic=dict(zip(manifest.column,manifest.semantic_type))
    semantic['RADurationDays_log1p']='numeric'
    semantic.pop('RADurationDays',None)
    return data,predictors,semantic


def make_preprocessor(data:pd.DataFrame,predictors:list[str],semantic:dict):
    numeric=[x for x in predictors if semantic[x]=='numeric']
    binary=[x for x in predictors if semantic[x]=='binary']
    categorical=[x for x in predictors if semantic[x]=='categorical']
    indicator_sources=[x for x in numeric+binary if data[x].isna().any()]
    transformers=[]
    if numeric:
        transformers.append(('numeric',Pipeline([
            ('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),
            ('scaler',StandardScaler()),
        ]),numeric))
    if binary:
        transformers.append(('binary',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True)),
        ]),binary))
    if categorical:
        for col in categorical:
            observed=set(data[col].dropna().astype(str).unique())
            allowed=set(CATEGORY_LEVELS[col])-{'Unknown'}
            if not observed.issubset(allowed):
                raise ValueError(f'Unexpected category in {col}: {observed-allowed}')
        categories=[CATEGORY_LEVELS[x] for x in categorical]
        transformers.append(('categorical',Pipeline([
            ('imputer',SimpleImputer(strategy='constant',fill_value='Unknown',keep_empty_features=True)),
            ('onehot',OneHotEncoder(categories=categories,drop='first',handle_unknown='ignore',sparse_output=True)),
        ]),categorical))
    if indicator_sources:
        transformers.append(('missing',MissingnessFlags(),indicator_sources))
    prep=ColumnTransformer(transformers,remainder='drop',verbose_feature_names_out=True)
    meta={'numeric':numeric,'binary':binary,'categorical':categorical,'missing':indicator_sources}
    return prep,meta


In [ ]:
# 6. Transformed-feature mapping for coefficients and linear SHAP
def transformed_feature_metadata(fitted_preprocessor,meta):
    transformed=[]; original=[]
    for col in meta['numeric']:
        transformed.append(f'numeric__{col}'); original.append('RADurationDays' if col=='RADurationDays_log1p' else col)
    for col in meta['binary']:
        transformed.append(f'binary__{col}'); original.append(col)
    if meta['categorical']:
        ohe=fitted_preprocessor.named_transformers_['categorical'].named_steps['onehot']
        names=ohe.get_feature_names_out(meta['categorical']).tolist()
        for name in names:
            source=next(col for col in meta['categorical'] if name.startswith(col+'_'))
            transformed.append('categorical__'+name); original.append(source)
    for col in meta['missing']:
        transformed.append(f'missing__{col}__missing')
        original.append('RADurationDays' if col=='RADurationDays_log1p' else col)
    return transformed,original

def dense(x):
    return x.toarray() if hasattr(x,'toarray') else np.asarray(x)

def grouped_linear_shap(fitted_pipeline,X_train,X_test,meta):
    prep=fitted_pipeline.named_steps['preprocess']; model=fitted_pipeline.named_steps['model']
    z_train=dense(prep.transform(X_train)); z_test=dense(prep.transform(X_test))
    coef=model.coef_.ravel(); background=z_train.mean(axis=0)
    shap_trans=(z_test-background)*coef
    transformed,original=transformed_feature_metadata(prep,meta)
    if len(transformed)!=len(coef):
        raise AssertionError((len(transformed),len(coef),transformed))
    decision=model.decision_function(z_test)
    base=float(model.intercept_[0]+background@coef)
    if np.max(np.abs(base+shap_trans.sum(axis=1)-decision))>1e-7:
        raise AssertionError('Linear SHAP additivity check failed.')
    unique=list(dict.fromkeys(original))
    grouped=np.column_stack([shap_trans[:,np.array(original)==x].sum(axis=1) for x in unique])
    return grouped,unique,pd.DataFrame({'transformed_feature':transformed,'original_feature':original,'coefficient':coef}),base


## Metrics and bootstrap functions

In [ ]:
# 7. Performance metrics, including calibration intercept and slope
def calibration_intercept_slope(y,p):
    p=np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6); y=np.asarray(y,dtype=float)
    x=logit(p); X=np.column_stack([np.ones(len(x)),x]); beta=np.array([0.,1.])
    for _ in range(50):
        mu=expit(X@beta); w=np.maximum(mu*(1-mu),1e-9)
        score=X.T@(y-mu); info=X.T@(w[:,None]*X)
        try: step=np.linalg.solve(info,score)
        except np.linalg.LinAlgError: return np.nan,np.nan
        beta_new=beta+step
        if np.max(np.abs(step))<1e-9: beta=beta_new; break
        beta=beta_new
    return float(beta[0]),float(beta[1])

def metric_dict(y,p):
    intercept,slope=calibration_intercept_slope(y,p)
    return {
        'auroc':roc_auc_score(y,p),
        'average_precision':average_precision_score(y,p),
        'log_loss':log_loss(y,p,labels=[0,1]),
        'brier':brier_score_loss(y,p),
        'calibration_intercept':intercept,
        'calibration_slope':slope,
    }

def threshold_metrics(y,p,threshold):
    pred=(np.asarray(p)>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        'threshold':threshold,'tn':tn,'fp':fp,'fn':fn,'tp':tp,
        'sensitivity':tp/(tp+fn) if tp+fn else np.nan,
        'specificity':tn/(tn+fp) if tn+fp else np.nan,
        'ppv':tp/(tp+fp) if tp+fp else np.nan,
        'npv':tn/(tn+fn) if tn+fn else np.nan,
    }

def stratified_bootstrap_indices(y,n_boot,seed):
    rng=np.random.default_rng(seed); y=np.asarray(y)
    neg=np.flatnonzero(y==0); pos=np.flatnonzero(y==1)
    for _ in range(n_boot):
        yield np.concatenate([rng.choice(neg,len(neg),replace=True),rng.choice(pos,len(pos),replace=True)])


## Prespecified models and tuning spaces

In [ ]:
# 8. Model specifications
@dataclass
class ModelSpec:
    name:str
    scope:str
    estimator:LogisticRegression
    param_grid:dict|None
    confirmatory:bool

# scikit-learn 1.8+ API: do not pass the deprecated penalty argument.
def ridge_estimator(solver='lbfgs'):
    return LogisticRegression(
        l1_ratio=0.0,solver=solver,max_iter=5000,random_state=MASTER_SEED)

def ordinary_estimator():
    return LogisticRegression(
        C=np.inf,l1_ratio=0.0,solver='lbfgs',max_iter=5000,random_state=MASTER_SEED)

def elastic_estimator():
    return LogisticRegression(
        l1_ratio=0.5,solver='saga',max_iter=10000,tol=1e-4,random_state=MASTER_SEED)

MODEL_SPECS=[
    ModelSpec('core_ridge','core',ridge_estimator(),{'model__C':RIDGE_C_GRID},True),
    ModelSpec('core_logistic','core',ordinary_estimator(),None,False),
    ModelSpec('broad_ridge','broad',ridge_estimator(),{'model__C':RIDGE_C_GRID},False),
    ModelSpec('broad_elastic_net','broad',elastic_estimator(),
              {'model__C':ENET_C_GRID,'model__l1_ratio':ENET_L1_GRID},False),
]

display(pd.DataFrame([{
    'model':s.name,'scope':s.scope,'confirmatory':s.confirmatory,
    'tuned':s.param_grid is not None,
    'grid_candidates':1 if s.param_grid is None else int(np.prod([len(x) for x in s.param_grid.values()])),
} for s in MODEL_SPECS]))


## Repeated nested cross-validation engine

In [ ]:
# 9. Fit one outcome with checkpointing
def fit_outcome(outcome:str):
    checkpoint=CHECKPOINT_DIR/f'{outcome}__{"quick" if QUICK_MODE else "full"}.pkl'
    if checkpoint.exists() and not FORCE_RECOMPUTE:
        with open(checkpoint,'rb') as f: saved=pickle.load(f)
        if saved.get('run_signature')==RUN_SIGNATURE:
            print('Loading valid checkpoint:',checkpoint)
            return saved['result']
        print('Ignoring checkpoint with a different signature:',checkpoint)

    datasets={}
    for scope in ['core','broad']:
        data,predictors,semantic=prepare_analysis_data(outcome,scope)
        datasets[scope]=(data,predictors,semantic)
    y=datasets['core'][0][outcome].astype(int).to_numpy()
    ids=datasets['core'][0]['VisitID'].to_numpy()
    if not np.array_equal(ids,datasets['broad'][0]['VisitID'].to_numpy()): raise AssertionError('Row mismatch')

    predictions=[]; hyperparameters=[]; fold_metrics=[]; coefficients=[]; fold_shap=[]; split_rows=[]
    shap_acc={}; shap_count={}; shap_features={}
    for spec in MODEL_SPECS:
        predictors=datasets[spec.scope][1]
        semantic=datasets[spec.scope][2]
        # Match ColumnTransformer output order: numeric, binary, then categorical.
        ordered=[x for typ in ['numeric','binary','categorical'] for x in predictors if semantic[x]==typ]
        originals=['RADurationDays' if x=='RADurationDays_log1p' else x for x in ordered]
        originals=list(dict.fromkeys(originals))
        shap_acc[spec.name]=np.zeros((len(y),len(originals)),dtype=float)
        shap_count[spec.name]=np.zeros(len(y),dtype=int)
        shap_features[spec.name]=originals

    outer=RepeatedStratifiedKFold(n_splits=OUTER_SPLITS,n_repeats=OUTER_REPEATS,random_state=MASTER_SEED)
    start=time.time()
    for split_index,(train_idx,test_idx) in enumerate(outer.split(np.zeros(len(y)),y)):
        repeat=split_index//OUTER_SPLITS; fold=split_index%OUTER_SPLITS
        inner=StratifiedKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=MASTER_SEED+1000*repeat+fold)
        for idx in test_idx:
            split_rows.append({'VisitID':ids[idx],'repeat':repeat,'fold':fold,'outcome':outcome})

        for spec in MODEL_SPECS:
            data,predictors,semantic=datasets[spec.scope]
            X=data[predictors]
            X_train,X_test=X.iloc[train_idx],X.iloc[test_idx]
            prep,meta=make_preprocessor(data,predictors,semantic)
            pipe=Pipeline([('preprocess',prep),('model',clone(spec.estimator))],memory=PIPELINE_MEMORY)
            # Grid candidates run in parallel; limit each candidate to one BLAS thread to
            # prevent CPU oversubscription in Colab and desktop environments.
            if spec.param_grid is None:
                with threadpool_limits(limits=1,user_api='blas'):
                    fitted=pipe.fit(X_train,y[train_idx])
                best_params={}
            else:
                search=GridSearchCV(pipe,spec.param_grid,scoring='neg_log_loss',cv=inner,
                                    refit=True,n_jobs=N_JOBS,pre_dispatch='2*n_jobs',
                                    error_score='raise',return_train_score=False)
                with threadpool_limits(limits=1,user_api='blas'):
                    search.fit(X_train,y[train_idx])
                fitted=search.best_estimator_; best_params=search.best_params_
                best_index=search.best_index_
                hyperparameters.append({
                    'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,
                    'best_C':best_params.get('model__C',np.nan),
                    'best_l1_ratio':best_params.get('model__l1_ratio',np.nan),
                    'best_inner_mean_log_loss':-search.cv_results_['mean_test_score'][best_index],
                    'best_inner_se_log_loss':search.cv_results_['std_test_score'][best_index]/math.sqrt(INNER_SPLITS),
                    'C_at_grid_boundary':best_params.get('model__C',None) in ({min(RIDGE_C_GRID),max(RIDGE_C_GRID)} if spec.name.endswith('ridge') else {min(ENET_C_GRID),max(ENET_C_GRID)}),
                })
            p=fitted.predict_proba(X_test)[:,1]
            for local,row_idx in enumerate(test_idx):
                predictions.append({'VisitID':ids[row_idx],'row_index':row_idx,'outcome':outcome,
                                    'model':spec.name,'repeat':repeat,'fold':fold,
                                    'y':y[row_idx],'probability':p[local]})
            fm=metric_dict(y[test_idx],p)
            fold_metrics.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,
                                 'n_test':len(test_idx),'events_test':int(y[test_idx].sum()),**fm})

            grouped,features,coef_table,base=grouped_linear_shap(fitted,X_train,X_test,meta)
            if features!=shap_features[spec.name]: raise AssertionError((features,shap_features[spec.name]))
            shap_acc[spec.name][test_idx,:]+=grouped
            shap_count[spec.name][test_idx]+=1
            fold_importance=np.mean(np.abs(grouped),axis=0)
            ranks=pd.Series(-fold_importance).rank(method='average').to_numpy()
            for feature,importance,rank in zip(features,fold_importance,ranks):
                fold_shap.append({'outcome':outcome,'model':spec.name,'repeat':repeat,'fold':fold,
                                  'feature':feature,'mean_abs_shap':importance,'rank':rank})
            coef_table['outcome']=outcome; coef_table['model']=spec.name
            coef_table['repeat']=repeat; coef_table['fold']=fold; coef_table['base_log_odds']=base
            coefficients.extend(coef_table.to_dict('records'))

        if (split_index+1)%max(1,OUTER_SPLITS)==0:
            elapsed=(time.time()-start)/60
            print(f'{outcome}: completed repeat {repeat+1}/{OUTER_REPEATS} ({elapsed:.1f} min)')

    shap_wide={}
    for spec in MODEL_SPECS:
        if not np.all(shap_count[spec.name]==OUTER_REPEATS):
            raise AssertionError(f'Unexpected SHAP counts for {spec.name}: {np.unique(shap_count[spec.name])}')
        avg=shap_acc[spec.name]/shap_count[spec.name][:,None]
        sw=pd.DataFrame(avg,columns=shap_features[spec.name])
        sw.insert(0,'VisitID',ids); sw.insert(1,'y',y)
        shap_wide[spec.name]=sw

    result={
        'predictions':pd.DataFrame(predictions),
        'hyperparameters':pd.DataFrame(hyperparameters),
        'fold_metrics':pd.DataFrame(fold_metrics),
        'coefficients':pd.DataFrame(coefficients),
        'fold_shap':pd.DataFrame(fold_shap),
        'shap_wide':shap_wide,
        'splits':pd.DataFrame(split_rows),
        'y':y,'ids':ids,
    }
    with open(checkpoint,'wb') as f: pickle.dump({'run_signature':RUN_SIGNATURE,'result':result},f,protocol=pickle.HIGHEST_PROTOCOL)
    print('Saved checkpoint:',checkpoint)
    return result


In [ ]:
# 10. Execute or resume all three outcomes
all_results={}
for outcome in OUTCOMES:
    display(Markdown(f'## Running `{outcome}`'))
    all_results[outcome]=fit_outcome(outcome)
print('All outcome pipelines completed.')


## Aggregate out-of-fold predictions and calculate performance

In [ ]:
# 11. Average the ten OOF probabilities per patient
all_predictions=pd.concat([r['predictions'] for r in all_results.values()],ignore_index=True)
all_predictions.to_csv(TABLE_DIR/'all_repeated_oof_predictions.csv',index=False,encoding='utf-8-sig')

avg_predictions=(all_predictions.groupby(['outcome','model','VisitID','y'],as_index=False)
                 .agg(probability=('probability','mean'),oof_repetitions=('probability','size'),
                      probability_sd_across_repeats=('probability','std')))
expected_repeats=OUTER_REPEATS
if not avg_predictions.oof_repetitions.eq(expected_repeats).all():
    raise AssertionError('Each patient must have one OOF prediction per repeat and model.')
avg_predictions.to_csv(TABLE_DIR/'patient_averaged_oof_predictions.csv',index=False,encoding='utf-8-sig')

performance_rows=[]
for (outcome,model),g in avg_predictions.groupby(['outcome','model'],sort=False):
    performance_rows.append({'outcome':outcome,'model':model,'n':len(g),'events':int(g.y.sum()),**metric_dict(g.y,g.probability)})
performance=pd.DataFrame(performance_rows)
performance.to_csv(TABLE_DIR/'point_estimate_performance.csv',index=False,encoding='utf-8-sig')
display(performance)


In [ ]:
# 12. Stratified patient-level bootstrap CIs and paired differences
bootstrap_metric_rows=[]; difference_rows=[]
HIGHER_BETTER={'auroc':True,'average_precision':True,'log_loss':False,'brier':False,
               'calibration_intercept':None,'calibration_slope':None}
for outcome in OUTCOMES:
    wide=avg_predictions[avg_predictions.outcome.eq(outcome)].pivot(index=['VisitID','y'],columns='model',values='probability').reset_index()
    y=wide.y.to_numpy()
    point={m:metric_dict(y,wide[m].to_numpy()) for m in MODEL_ORDER}
    boot_values={m:{k:[] for k in point[m]} for m in MODEL_ORDER}
    diff_values={m:{k:[] for k in point[m]} for m in MODEL_ORDER if m!='core_ridge'}
    for idx in stratified_bootstrap_indices(y,BOOTSTRAPS,MASTER_SEED+OUTCOMES.index(outcome)*100):
        bm={m:metric_dict(y[idx],wide[m].to_numpy()[idx]) for m in MODEL_ORDER}
        for m in MODEL_ORDER:
            for k,v in bm[m].items(): boot_values[m][k].append(v)
        for m in diff_values:
            for k in diff_values[m]:
                if HIGHER_BETTER[k] is True: val=bm[m][k]-bm['core_ridge'][k]
                elif HIGHER_BETTER[k] is False: val=bm['core_ridge'][k]-bm[m][k]
                elif k=='calibration_intercept': val=abs(bm['core_ridge'][k])-abs(bm[m][k])
                else: val=abs(1-bm['core_ridge'][k])-abs(1-bm[m][k])
                diff_values[m][k].append(val)
    for m in MODEL_ORDER:
        for k,vals in boot_values[m].items():
            vals=np.asarray(vals,float)
            bootstrap_metric_rows.append({
                'outcome':outcome,'model':m,'metric':k,'estimate':point[m][k],
                'ci_low':np.nanpercentile(vals,2.5),'ci_high':np.nanpercentile(vals,97.5),
                'bootstrap_resamples':BOOTSTRAPS,
            })
    for m in diff_values:
        for k,vals in diff_values[m].items():
            vals=np.asarray(vals,float)
            if HIGHER_BETTER[k] is True: point_diff=point[m][k]-point['core_ridge'][k]
            elif HIGHER_BETTER[k] is False: point_diff=point['core_ridge'][k]-point[m][k]
            elif k=='calibration_intercept': point_diff=abs(point['core_ridge'][k])-abs(point[m][k])
            else: point_diff=abs(1-point['core_ridge'][k])-abs(1-point[m][k])
            difference_rows.append({
                'outcome':outcome,'comparator':m,'reference':'core_ridge','metric':k,
                'improvement_over_core':point_diff,
                'ci_low':np.nanpercentile(vals,2.5),'ci_high':np.nanpercentile(vals,97.5),
                'positive_value_favors':'comparator',
            })

performance_ci=pd.DataFrame(bootstrap_metric_rows)
paired_differences=pd.DataFrame(difference_rows)
performance_ci.to_csv(TABLE_DIR/'performance_with_bootstrap_ci.csv',index=False,encoding='utf-8-sig')
paired_differences.to_csv(TABLE_DIR/'paired_model_differences.csv',index=False,encoding='utf-8-sig')
display(performance_ci)
display(paired_differences)


## Threshold grids, calibration, discrimination, and stability

In [ ]:
# 13. Prespecified threshold grid; no threshold is selected as primary
threshold_rows=[]
for (outcome,model),g in avg_predictions.groupby(['outcome','model']):
    for threshold in np.arange(0.05,0.51,0.05):
        threshold_rows.append({'outcome':outcome,'model':model,**threshold_metrics(g.y,g.probability,float(threshold))})
threshold_table=pd.DataFrame(threshold_rows)
threshold_table.to_csv(TABLE_DIR/'threshold_grid_metrics.csv',index=False,encoding='utf-8-sig')
display(threshold_table.head(20))


In [ ]:
# 14. Save fold/repetition stability and hyperparameter stability
fold_metrics=pd.concat([r['fold_metrics'] for r in all_results.values()],ignore_index=True)
hyperparameters=pd.concat([r['hyperparameters'] for r in all_results.values()],ignore_index=True)
coefficients=pd.concat([r['coefficients'] for r in all_results.values()],ignore_index=True)
fold_shap=pd.concat([r['fold_shap'] for r in all_results.values()],ignore_index=True)
splits=pd.concat([r['splits'] for r in all_results.values()],ignore_index=True)
for frame,name in [(fold_metrics,'outer_fold_metric_stability.csv'),(hyperparameters,'selected_hyperparameters.csv'),
                   (coefficients,'outer_fold_coefficients.csv'),(fold_shap,'outer_fold_shap_stability.csv'),
                   (splits,'saved_outer_split_assignments.csv')]:
    frame.to_csv(TABLE_DIR/name,index=False,encoding='utf-8-sig')

fold_summary=(fold_metrics.groupby(['outcome','model'])[['auroc','average_precision','log_loss','brier','calibration_intercept','calibration_slope']]
              .agg(['median','std','min','max']).reset_index())
fold_summary.columns=['__'.join([str(x) for x in col if str(x)]) for col in fold_summary.columns]
fold_summary.to_csv(TABLE_DIR/'outer_fold_metric_stability_summary.csv',index=False,encoding='utf-8-sig')
display(fold_summary)

if not hyperparameters.empty:
    boundary_summary=(hyperparameters.groupby(['outcome','model'],as_index=False)
                      .agg(boundary_hit_fraction=('C_at_grid_boundary','mean'),
                           median_best_C=('best_C','median'),
                           min_best_C=('best_C','min'),max_best_C=('best_C','max')))
    boundary_summary.to_csv(TABLE_DIR/'hyperparameter_boundary_summary.csv',index=False,encoding='utf-8-sig')
    display(boundary_summary)


In [ ]:
# 15. Calibration, ROC, and precision-recall plots
for outcome in OUTCOMES:
    fig,axes=plt.subplots(1,3,figsize=(18,5.2))
    sub=avg_predictions[avg_predictions.outcome.eq(outcome)]
    for model in MODEL_ORDER:
        g=sub[sub.model.eq(model)].sort_values('probability')
        y=g.y.to_numpy(); p=g.probability.to_numpy()
        # Flexible descriptive isotonic smoother on cross-fitted predictions.
        iso=IsotonicRegression(out_of_bounds='clip').fit(p,y)
        grid=np.linspace(max(0,p.min()),min(1,p.max()),150)
        axes[0].plot(grid,iso.predict(grid),label=model)
        fpr,tpr,_=roc_curve(y,p); axes[1].plot(fpr,tpr,label=f'{model} ({roc_auc_score(y,p):.3f})')
        precision,recall,_=precision_recall_curve(y,p); axes[2].plot(recall,precision,label=f'{model} ({average_precision_score(y,p):.3f})')
    axes[0].plot([0,1],[0,1],'k--',alpha=.7); axes[0].set(xlabel='Predicted probability',ylabel='Observed proportion',title='OOF calibration smoother')
    axes[1].plot([0,1],[0,1],'k--',alpha=.5); axes[1].set(xlabel='False-positive rate',ylabel='True-positive rate',title='ROC curves')
    axes[2].axhline(sub.y.mean(),color='k',ls='--',alpha=.5); axes[2].set(xlabel='Recall',ylabel='Precision',title='Precision–recall curves')
    for ax in axes: ax.legend(fontsize=8)
    fig.suptitle(outcome,fontsize=15,fontweight='bold'); plt.tight_layout()
    plt.savefig(FIGURE_DIR/f'{outcome}__performance_curves.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close()


## Out-of-fold SHAP and coefficient stability

In [ ]:
# 16. Save patient-level OOF SHAP and create global/stability summaries
shap_summary_rows=[]; shap_stability_rows=[]
for outcome,result in all_results.items():
    for model,sw in result['shap_wide'].items():
        sw.to_csv(TABLE_DIR/f'{outcome}__{model}__patient_oof_shap.csv',index=False,encoding='utf-8-sig')
        feature_cols=[x for x in sw.columns if x not in {'VisitID','y'}]
        fold_sub=result['fold_shap'][result['fold_shap'].model.eq(model)]
        for feature in feature_cols:
            vals=sw[feature].to_numpy()
            fs=fold_sub[fold_sub.feature.eq(feature)]
            shap_summary_rows.append({
                'outcome':outcome,'model':model,'feature':feature,
                'mean_abs_shap':np.mean(np.abs(vals)),'mean_signed_shap':np.mean(vals),
                'median_abs_shap':np.median(np.abs(vals)),
                'positive_shap_fraction':np.mean(vals>0),
            })
            shap_stability_rows.append({
                'outcome':outcome,'model':model,'feature':feature,
                'median_fold_rank':fs['rank'].median(),'rank_iqr':fs['rank'].quantile(.75)-fs['rank'].quantile(.25),
                'top5_fold_frequency':np.mean(fs['rank']<=5),'top10_fold_frequency':np.mean(fs['rank']<=10),
                'fold_mean_abs_shap_sd':fs['mean_abs_shap'].std(),
            })
shap_summary=pd.DataFrame(shap_summary_rows)
shap_stability=pd.DataFrame(shap_stability_rows)
shap_summary['global_rank']=shap_summary.groupby(['outcome','model'])['mean_abs_shap'].rank(ascending=False,method='min')
shap_summary=shap_summary.merge(shap_stability,on=['outcome','model','feature'],how='left')
shap_summary.to_csv(TABLE_DIR/'global_oof_shap_summary.csv',index=False,encoding='utf-8-sig')
display(shap_summary.sort_values(['outcome','model','global_rank']).groupby(['outcome','model']).head(12))


In [ ]:
# 17. Transformed-coefficient sign and selection stability
coef_summary=(coefficients.groupby(['outcome','model','transformed_feature','original_feature'],as_index=False)
              .agg(median_coefficient=('coefficient','median'),
                   coefficient_q025=('coefficient',lambda x:np.quantile(x,.025)),
                   coefficient_q975=('coefficient',lambda x:np.quantile(x,.975)),
                   positive_fraction=('coefficient',lambda x:np.mean(x>0)),
                   zero_fraction=('coefficient',lambda x:np.mean(np.isclose(x,0,atol=1e-10))),
                   n_outer_models=('coefficient','size')))
coef_summary['sign_stability']=np.maximum(coef_summary.positive_fraction,1-coef_summary.positive_fraction)
coef_summary.to_csv(TABLE_DIR/'coefficient_sign_selection_stability.csv',index=False,encoding='utf-8-sig')
display(coef_summary.sort_values(['outcome','model','sign_stability'],ascending=[True,True,False]).head(40))


In [ ]:
# 18. SHAP magnitude and rank-stability plots
for outcome in OUTCOMES:
    for model in ['core_ridge','broad_ridge','broad_elastic_net']:
        top=(shap_summary[(shap_summary.outcome.eq(outcome))&(shap_summary.model.eq(model))]
             .sort_values('mean_abs_shap',ascending=False).head(15).sort_values('mean_abs_shap'))
        plt.figure(figsize=(9,6.5))
        plt.barh(top.feature,top.mean_abs_shap,color='#146C72')
        plt.xlabel('Mean absolute out-of-fold SHAP value (log-odds scale)')
        plt.ylabel('Original clinical variable')
        plt.title(f'{outcome}: {model} OOF feature importance')
        plt.tight_layout(); plt.savefig(FIGURE_DIR/f'{outcome}__{model}__oof_shap.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close()


## Final audit and scientific status

In [ ]:
# 19. Automated validation of the complete modeling run
audit_rows=[]
def check(cid,description,passed,details=''):
    audit_rows.append({'check_id':cid,'description':description,'status':'PASS' if bool(passed) else 'FAIL','details':details})

check('M001','Execution mode explicitly recorded',True,
      'QUICK SOFTWARE TEST — results are not scientific' if QUICK_MODE else 'FULL scientific mode')
check('M002','Three outcomes completed',set(all_results)==set(OUTCOMES))
check('M003','Four prespecified models completed',set(avg_predictions.model)==set(MODEL_ORDER))
check('M004','Every patient has expected OOF repetitions',avg_predictions.oof_repetitions.eq(OUTER_REPEATS).all())
check('M005','No missing averaged probabilities',avg_predictions.probability.notna().all())
check('M006','All probabilities within zero and one',avg_predictions.probability.between(0,1,inclusive='both').all())
check('M007','All outcome-model combinations contain 820 patients',avg_predictions.groupby(['outcome','model']).size().eq(820).all())
check('M008','All performance metrics finite',np.isfinite(performance.select_dtypes('number')).all().all())
check('M009','Bootstrap count matches execution mode',BOOTSTRAPS==(50 if QUICK_MODE else 2000))
check('M010','Outer design matches execution mode',
      (OUTER_SPLITS,OUTER_REPEATS)==((3,1) if QUICK_MODE else (5,10)))
check('M011','Inner design matches execution mode',INNER_SPLITS==(2 if QUICK_MODE else 5))
check('M012','Decision curve not run without clinical action',True,'No net-benefit claim generated.')
check('M013','SHAP generated strictly from outer-test observations',all(
    len(r['shap_wide'][m])==820 for r in all_results.values() for m in MODEL_ORDER))
check('M014','SHAP repetition counts complete',True,'Validated inside each outcome checkpoint.')
check('M015','No model-family winner was selected',True,'Core ridge remains prespecified confirmatory model.')
check('M016','External validation not claimed',True,'Internal validation only.')
check('M017','No threshold selected as primary',True,'Threshold grid only.')
check('M018','All compared models use identical outer splits within outcome',True,'Single outer split loop per outcome.')
check('M019','Hyperparameter tuning used log loss',True,'GridSearchCV scoring=neg_log_loss.')
check('M020','Class weighting, SMOTE, and undersampling absent from primary run',True,'All estimators use natural prevalence.')

model_audit=pd.DataFrame(audit_rows)
model_audit.to_csv(TABLE_DIR/'stage3b_validation_audit.csv',index=False,encoding='utf-8-sig')
display(model_audit)
if not model_audit.status.eq('PASS').all():
    raise AssertionError(model_audit[model_audit.status.eq('FAIL')].to_string(index=False))
print(f'ALL {len(model_audit)} STAGE 3B CHECKS PASSED')


In [ ]:
# 20. Artifact registry and compact run summary
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
artifacts=[]
for folder in [TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,OUT_DIR]:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.name!='artifact_registry.csv':
            artifacts.append({'filename':p.name,'path':str(p),'size_bytes':p.stat().st_size,'sha256':file_hash(p)})
artifact_registry=pd.DataFrame(artifacts).drop_duplicates('path')
artifact_registry.to_csv(TABLE_DIR/'artifact_registry.csv',index=False,encoding='utf-8-sig')
run_summary={
    'run_signature':RUN_SIGNATURE,'mode':'quick' if QUICK_MODE else 'full',
    'outcomes':OUTCOMES,'models':MODEL_ORDER,
    'outer_splits':OUTER_SPLITS,'outer_repeats':OUTER_REPEATS,'inner_splits':INNER_SPLITS,
    'bootstrap_resamples':BOOTSTRAPS,'all_audits_passed':bool(model_audit.status.eq('PASS').all()),
    'confirmatory_model':'core_ridge','external_validation':False,
}
with open(OUT_DIR/'stage3b_run_summary.json','w',encoding='utf-8') as f: json.dump(run_summary,f,indent=2)
display(artifact_registry[['filename','size_bytes']])


## Final cell — clean HTML export using validated checkpoints

Save this notebook as:

```text
the authorized RA_PROJECT_DIR\03B_modeling_validation.ipynb
```

After the first full run creates signed checkpoints, the final cell launches a clean execution. The clean process verifies and loads those checkpoints, regenerates all metrics and plots, and exports:

```text
the authorized RA_PROJECT_DIR\modeling_results\report\Stage3B_modeling_validation_report.html
```


In [ ]:
# 21. FINAL CELL: clean execution and HTML export
if os.environ.get('RA_STAGE3B_EXPORT_MODE')=='1':
    print('Stage 3B HTML subprocess mode: recursive export intentionally skipped.')
else:
    candidates=[PROJECT_DIR/NOTEBOOK_NAME,COLAB_DRIVE_PROJECT/NOTEBOOK_NAME,
                COLAB_UPLOAD_PROJECT/NOTEBOOK_NAME,ENV_PROJECT/NOTEBOOK_NAME,
                Path.cwd()/NOTEBOOK_NAME,Path.cwd()/'ra_analysis'/NOTEBOOK_NAME]
    notebook_path=next((p for p in candidates if p.exists()),None)
    if notebook_path is None:
        raise FileNotFoundError(f'Save the notebook as {NOTEBOOK_NAME} before export.')
    env=os.environ.copy(); env['RA_STAGE3B_EXPORT_MODE']='1'; env.pop('RA_STAGE3B_QUICK_TEST',None)
    cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute',
         '--ExecutePreprocessor.timeout=1800','--ExecutePreprocessor.kernel_name=python3',
         '--HTMLExporter.embed_images=True','--output','Stage3B_modeling_validation_report.html',
         '--output-dir',str(REPORT_DIR),str(notebook_path)]
    print('Executing:',' '.join(cmd))
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    print(result.stdout)
    if result.returncode!=0:
        print(result.stderr); raise RuntimeError(f'HTML export failed with exit code {result.returncode}')
    report=REPORT_DIR/'Stage3B_modeling_validation_report.html'
    if not report.exists(): raise FileNotFoundError(report)
    print('HTML report created:',report.resolve())
